# Quantify pseudohyphal growth in single-well images

This notebook measures the bright colony center and the surrounding intermediate-intensity growth in images of individual wells. The input images should already have been separated from the original 96-well plate image.

For each well, the workflow:

1. converts the image to grayscale;
2. estimates background and bright-center intensity thresholds;
3. isolates and fills the largest bright-center region (the main body of the colony);
4. measures the intermediate-intensity outer region (pseudohyphae);
5. calculates **Social Growth (%) = 100 × outer-region pixels / center pixels**; and
6. saves a processed mask and an Excel summary.

white: colony; gray: pseudohyphae; black: background

A larger Social Growth (%) means that the measured outer region is larger relative to the bright center. Images with no detected center receive a missing (`NaN`) score because the ratio cannot be calculated.

Threshold-based segmentation is sensitive to lighting, contrast, dust, and off-center colonies. Review processed masks across each new imaging condition—especially wells with unusually high or low scores—before using the measurements for downstream analysis.

## 1. Imports and configuration

Update `INPUT_FOLDER` for the folder containing the single-well images. Processed masks and the Excel results file are written to `OUTPUT_FOLDER`.

The default settings reproduce the automatic-threshold workflow used in this notebook. If automatic bright-center detection is unsuitable for a dataset, set `WHITE_THRESHOLD_OVERRIDE` to a value from 0 to 1 (for example, `0.43`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import binary_fill_holes, gaussian_filter1d
from skimage import color, io, measure, morphology
from skimage.filters import threshold_otsu
from skimage.util import img_as_float

# Input: one image per well (PNG, JPEG, TIFF, BMP, or GIF).
INPUT_FOLDER = Path(r"D:\aaa_murphy_lab\Armaan_images\PSH\sorted\HMY355D_SLAD_5D_3_crop\yes")

# Output: processed masks and one Excel summary file.
OUTPUT_FOLDER = Path(r"D:\aaa_murphy_lab\Armaan_images\PSH\eclipse\HMY355D_SLAD_5D_3_crop")
RESULTS_FILE = OUTPUT_FOLDER / "HMY355D_SLAD_5D_3_crop_social_growth.xlsx"

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".gif"}

# Segmentation settings.
BACKGROUND_OFFSET = 0.01
WHITE_THRESHOLD_OFFSET = 0.045  
CLOSING_RADIUS = 6
WHITE_THRESHOLD_OVERRIDE = None  # Use None for automatic detection or a value from 0 to 1.


## 2. Segmentation and measurement functions

The background threshold is the first local minimum after the main peak of a smoothed grayscale histogram. The bright-center threshold is calculated with Otsu's method using pixels above the background threshold. The offsets in the configuration cell make these thresholds slightly less restrictive.

Only the largest connected bright region is treated as the colony center. Morphological closing and hole filling make that center region continuous. All non-background pixels outside the center are counted as outer growth.

In [ ]:
def load_grayscale(image_path):
    """Load an image and return grayscale pixel intensities from 0 to 1."""
    image = io.imread(image_path)

    if image.ndim == 3 and image.shape[-1] == 4:
        image = color.rgba2rgb(image)

    if image.ndim == 3:
        image_gray = color.rgb2gray(image)
    elif image.ndim == 2:
        image_gray = img_as_float(image)
    else:
        raise ValueError(f"Unsupported image shape {image.shape} for {image_path}")

    return np.clip(image_gray.astype(float), 0, 1)


def find_background_threshold(image_gray, offset=BACKGROUND_OFFSET):
    """Estimate the upper edge of the dark-background intensity distribution."""
    pixel_values = image_gray.ravel()
    histogram, bin_edges = np.histogram(pixel_values, bins=256, range=(0, 1))
    smoothed_histogram = gaussian_filter1d(histogram.astype(float), sigma=2)
    peak_index = int(np.argmax(smoothed_histogram))

    threshold = None
    for index in range(peak_index + 1, len(smoothed_histogram) - 1):
        is_local_minimum = (
            smoothed_histogram[index] <= smoothed_histogram[index - 1]
            and smoothed_histogram[index] <= smoothed_histogram[index + 1]
        )
        if is_local_minimum:
            threshold = bin_edges[index]
            break

    if threshold is None:
        threshold = np.percentile(pixel_values, 90)

    return float(np.clip(threshold - offset, 0, 1))


def find_white_threshold(
    image_gray,
    background_threshold,
    offset=WHITE_THRESHOLD_OFFSET,
):
    """Estimate the minimum intensity of the bright colony center."""
    candidate_pixels = image_gray[image_gray > background_threshold]

    if candidate_pixels.size < 10:
        return float(min(background_threshold + 0.1, 1.0))

    threshold = threshold_otsu(candidate_pixels)
    return float(np.clip(threshold - offset, 0, 1))


def segment_well(
    image_path,
    closing_radius=CLOSING_RADIUS,
    white_threshold_override=WHITE_THRESHOLD_OVERRIDE,
):
    """Segment one well and return its masks, thresholds, and measurements."""
    image_path = Path(image_path)
    image_gray = load_grayscale(image_path)
    background_threshold = find_background_threshold(image_gray)

    if white_threshold_override is None:
        white_threshold = find_white_threshold(image_gray, background_threshold)
    else:
        white_threshold = float(white_threshold_override)
        if not 0 <= white_threshold <= 1:
            raise ValueError("WHITE_THRESHOLD_OVERRIDE must be between 0 and 1.")

    if white_threshold <= background_threshold:
        raise ValueError(
            f"White threshold ({white_threshold:.3f}) must be greater than "
            f"background threshold ({background_threshold:.3f}) for {image_path.name}."
        )

    bright_mask = image_gray > white_threshold
    labeled_mask = measure.label(bright_mask, connectivity=2)
    regions = measure.regionprops(labeled_mask)

    if regions:
        largest_region = max(regions, key=lambda region: region.area)
        center_mask = labeled_mask == largest_region.label
        center_mask = morphology.binary_closing(
            center_mask, morphology.disk(closing_radius)
        )
        center_mask = binary_fill_holes(center_mask)
    else:
        center_mask = np.zeros_like(bright_mask, dtype=bool)

    non_background_mask = image_gray >= background_threshold
    outer_mask = non_background_mask & ~center_mask

    processed_mask = np.zeros_like(image_gray, dtype=float)
    processed_mask[outer_mask] = 0.5
    processed_mask[center_mask] = 0.8

    center_pixels = int(center_mask.sum())
    outer_pixels = int(outer_mask.sum())
    social_growth = (
        np.nan if center_pixels == 0 else 100 * outer_pixels / center_pixels
    )

    return {
        "image_name": image_path.name,
        "image_gray": image_gray,
        "center_mask": center_mask,
        "outer_mask": outer_mask,
        "processed_mask": processed_mask,
        "center_pixels": center_pixels,
        "outer_pixels": outer_pixels,
        "social_growth": social_growth,
        "background_threshold": background_threshold,
        "white_threshold": white_threshold,
    }


def plot_segmentation(segmentation):
    """Display one well, its final mask, and the thresholds used."""
    image_gray = segmentation["image_gray"]
    figure, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(image_gray, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Original well")
    axes[0].axis("off")

    axes[1].imshow(segmentation["processed_mask"], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Segmented mask")
    axes[1].axis("off")

    axes[2].hist(image_gray.ravel(), bins=256, range=(0, 1), color="gray")
    axes[2].axvline(
        segmentation["background_threshold"],
        color="tab:blue",
        linestyle="--",
        label="Background threshold",
    )
    axes[2].axvline(
        segmentation["white_threshold"],
        color="tab:orange",
        linestyle="--",
        label="Center threshold",
    )
    axes[2].set_title("Intensity histogram")
    axes[2].set_xlabel("Grayscale intensity")
    axes[2].set_ylabel("Pixel count")
    axes[2].legend()

    figure.suptitle(segmentation["image_name"])
    figure.tight_layout()
    plt.show()


def save_processed_mask(segmentation, output_folder):
    """Save a segmented mask as an 8-bit grayscale PNG."""
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    output_path = output_folder / segmentation["image_name"]
    output_image = (segmentation["processed_mask"] * 255).astype(np.uint8)
    io.imsave(output_path, output_image, check_contrast=False)
    return output_path


def process_images_in_folder(input_folder, output_folder, results_file):
    """Analyze all supported images in one folder and save the results."""
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    results_file = Path(results_file)

    if not input_folder.is_dir():
        raise FileNotFoundError(f"Input folder not found: {input_folder}")

    image_paths = sorted(
        path
        for path in input_folder.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No supported images found in: {input_folder}")

    rows = []
    for image_path in image_paths:
        segmentation = segment_well(image_path)
        save_processed_mask(segmentation, output_folder)
        rows.append(
            {
                "Image Name": segmentation["image_name"],
                "Social Growth %": segmentation["social_growth"],
                "Center Pixels": segmentation["center_pixels"],
                "Outer Pixels": segmentation["outer_pixels"],
                "Background Threshold": segmentation["background_threshold"],
                "White Threshold": segmentation["white_threshold"],
            }
        )
        print(
            f"{image_path.name} | "
            f"background={segmentation['background_threshold']:.3f}, "
            f"center={segmentation['white_threshold']:.3f}, "
            f"social growth={segmentation['social_growth']:.2f}%"
        )

    results = pd.DataFrame(rows)
    output_folder.mkdir(parents=True, exist_ok=True)
    results.to_excel(results_file, index=False)
    return results


## 3. Quality-check one well

Before processing a complete folder, inspect one representative image. The plot compares the original well with its segmented mask and shows the two intensity thresholds. Confirm that the light-gray area matches the colony center and the gray area matches the surrounding growth.

By default, the first supported image in `INPUT_FOLDER` is used. Replace `SAMPLE_IMAGE` with a specific file path if another well would be more representative.

In [ ]:
sample_images = sorted(
    path
    for path in INPUT_FOLDER.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)
if not sample_images:
    raise FileNotFoundError(f"No supported images found in: {INPUT_FOLDER}")

SAMPLE_IMAGE = sample_images[0]
preview = segment_well(SAMPLE_IMAGE)
plot_segmentation(preview)

print(f"Image: {preview['image_name']}")
print(f"Center pixels: {preview['center_pixels']:,}")
print(f"Outer pixels: {preview['outer_pixels']:,}")
print(f"Social Growth: {preview['social_growth']:.2f}%")
print(f"Background threshold: {preview['background_threshold']:.3f}")
print(f"Center threshold: {preview['white_threshold']:.3f}")


## 4. Process every well in the input folder

Run this cell after the single-image quality check looks reasonable. It analyzes each supported image directly inside `INPUT_FOLDER`, saves one processed mask per well in `OUTPUT_FOLDER`, and writes all measurements to `RESULTS_FILE`. Existing files with the same names are replaced.

In [ ]:
results = process_images_in_folder(
    input_folder=INPUT_FOLDER,
    output_folder=OUTPUT_FOLDER,
    results_file=RESULTS_FILE,
)

print(f"\nProcessed {len(results)} well image(s).")
print(f"Processed masks: {OUTPUT_FOLDER}")
print(f"Excel results: {RESULTS_FILE}")
results.head()
